## Elo

Expected score for team A:

$$
E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}
$$

Rating update:

$$
R_A^{\text{new}} = R_A + K(S_A - E_A)
$$

where:

- $R_A, R_B$ the current Elo ratings
- $S_A = 1$ for a win $0.5$ for a draw $0$ for a loss
- $K$ factor controls how quickly ratings respond to new results (needs to be chosen)
    - could use higher K for CL matches to reflect less data. eg K = 20 for domestic leagues, $K = 30$ or $40$ for CL matches
    - could use goal diff multiplier too, to separate large wins from same wins eg $G = 1$ if 0 or 1 GD, $G=1.5$ if gd is 2, G = some function of gd if greater than 3 (tbc)
    - can use higher K in early seasons for faster standardisation


In [64]:
import sqlite3
import pandas as pd

connection = sqlite3.connect("../data/processed/football.db")

matches = pd.read_sql_query(
    "SELECT * FROM matches",
    connection
)

print(matches.head())
matches.dtypes

   match_id        date   season competition  home_team_id  away_team_id  \
0         1  2015-06-30  2015/16          CL            14           384   
1         2  2015-06-30  2015/16          CL           184             8   
2         3  2015-06-30  2015/16          CL           326           104   
3         4  2015-07-01  2015/16          CL           106           503   
4         5  2015-07-07  2015/16          CL             8           184   

     home_team_name    away_team_name  home_goals  away_goals      source  
0            Pyunik           Folgore           2           1  footystats  
1  Lincoln Red Imps   FC Santa Coloma           0           0  footystats  
2         Crusaders   Levadia Tallinn           0           0  footystats  
3      B36 Torshavn    The New Saints           1           2  footystats  
4   FC Santa Coloma  Lincoln Red Imps           1           2  footystats  


match_id          int64
date                str
season              str
competition         str
home_team_id      int64
away_team_id      int64
home_team_name      str
away_team_name      str
home_goals        int64
away_goals        int64
source              str
dtype: object

In [63]:
# first pass of computing elos. all initialised at 1500, K = 20

team_ids = pd.unique(
    matches[["home_team_id", "away_team_id"]].values.ravel()
)

elo = {team_id: 1500 for team_id in team_ids}

K = 20

for _, match in matches.iterrows():
    S_A = 1 if match['home_goals'] > match['away_goals'] else 0.5 if match['home_goals'] == match['away_goals'] else 0
    S_B = 1 if match['home_goals'] < match['away_goals'] else 0.5 if match['home_goals'] == match['away_goals'] else 0

    current_home_rating = elo[match['home_team_id']]
    current_away_rating = elo[match['away_team_id']]

    Q_A = 10 ** (current_home_rating / 400)
    Q_B = 10 ** (current_away_rating / 400)

    expected_home_score = Q_A / (Q_A + Q_B)
    expected_away_score = Q_B / (Q_A + Q_B)

    updated_home_rating = current_home_rating + K * (S_A - expected_home_score)
    updated_away_rating = current_away_rating + K * (S_B - expected_away_score)

    elo[match['home_team_id']] = updated_home_rating
    elo[match['away_team_id']] = updated_away_rating

elo_df = pd.DataFrame(
    list(elo.items()),
    columns=["team_id", "elo"]
)

teams = pd.read_sql_query(
    "SELECT team_id, canonical_name FROM teams",
    connection
)

elo_df = elo_df.merge(
    teams,
    on='team_id',
    how='left'
)

elo_df = elo_df.sort_values('elo', ascending=False)

top_20 = elo_df.head(20)

mid_start = len(elo_df) // 2 - 10
mid_20 = elo_df.iloc[mid_start:mid_start + 20]

bottom_20 = elo_df.tail(20)

display(top_20)
display(mid_20)
display(bottom_20)

,team_id,elo,canonical_name
133,156,1898.222312,Bayern Munich
106,70,1855.853015,Arsenal
78,140,1826.271747,Paris Saint-Germain
231,421,1824.195152,Barcelona
152,359,1799.981550,Porto
136,363,1799.973935,Sporting CP
237,440,1798.621510,Real Madrid
220,240,1789.043630,Inter Milan
164,343,1781.796897,Benfica
124,310,1779.705496,PSV Eindhoven


,team_id,elo,canonical_name
249,113,1490.699658,SJK Seinajoki
461,408,1490.683694,TSC Backa Topola
387,449,1490.522091,Djurgarden
485,466,1490.245146,Bodrumspor
472,274,1490.165386,Panevezys
388,281,1490.018860,Floriana
394,56,1490.016550,Lokomotiva Zagreb
478,261,1490.000000,Ordabasy
22,291,1490.000000,Rudar Pljevlja
32,273,1490.000000,Ventspils


,team_id,elo,canonical_name
105,306,1369.170694,Heracles Almelo
145,167,1368.782822,Hannover 96
276,352,1364.329396,Feirense
367,93,1360.999983,Sheffield United
381,28,1360.175358,Beerschot
283,248,1358.876268,Pescara
144,160,1358.782196,Darmstadt 98
321,251,1358.215346,SPAL
266,33,1357.511345,Eupen
191,208,1356.459737,Platanias


From the first pass ($K = 20$, inital = 1500), we can see that:

1. Roughly speaking, "better" teams are higher, and "worse" teams are lower
2. The ordering is by no means conclusive, in fact there are several strong teams that are far lower than we'd expect and vice versa (eg Fenerbahce v high, Liverpool v low)

Why?

1. There is no separation between domestic games and CL games. Hence the inter league standardisation is off. Hence we don't know if dominating Portugese leagues is equivalent to dominating English leagues.

How to fix?

1. Consider starting ELOs differing based on the performance of each league represented in the CL
2. Consider K weighting stronger for CL games

Next: See CL elos to identify how things look in that table.



In [65]:
cl_elo = {}

K = 20

for _, match in matches[matches["competition"] == "CL"].iterrows():

    if match["home_team_id"] not in cl_elo:
        cl_elo[match["home_team_id"]] = 1500

    if match["away_team_id"] not in cl_elo:
        cl_elo[match["away_team_id"]] = 1500

    S_A = 1 if match["home_goals"] > match["away_goals"] else 0.5 if match["home_goals"] == match["away_goals"] else 0
    S_B = 1 if match["home_goals"] < match["away_goals"] else 0.5 if match["home_goals"] == match["away_goals"] else 0

    current_home_rating = cl_elo[match["home_team_id"]]
    current_away_rating = cl_elo[match["away_team_id"]]

    Q_A = 10 ** (current_home_rating / 400)
    Q_B = 10 ** (current_away_rating / 400)

    expected_home_score = Q_A / (Q_A + Q_B)
    expected_away_score = Q_B / (Q_A + Q_B)

    updated_home_rating = current_home_rating + K * (S_A - expected_home_score)
    updated_away_rating = current_away_rating + K * (S_B - expected_away_score)

    cl_elo[match["home_team_id"]] = updated_home_rating
    cl_elo[match["away_team_id"]] = updated_away_rating

cl_elo_df = pd.DataFrame(
    list(cl_elo.items()),
    columns=["team_id", "elo"]
)

teams = pd.read_sql_query(
    "SELECT DISTINCT t.team_id, t.canonical_name, a.league FROM teams AS t JOIN team_aliases AS a ON t.team_id = a.team_id WHERE a.source = 'football_data'",
    connection
)

cl_elo_df = cl_elo_df.merge(
    teams,
    on="team_id",
    how="left"
)

cl_elo_df = cl_elo_df.sort_values("elo", ascending=False)

cl_elo_df.head(20)

,team_id,elo,canonical_name,league
72,156,1762.763540,Bayern Munich,D1
62,440,1715.777908,Real Madrid,SP1
58,140,1713.458863,Paris Saint-Germain,F1
69,70,1682.200287,Arsenal,E0
56,87,1681.792475,Manchester City,E0
129,85,1674.385099,Liverpool,E0
74,421,1669.300790,Barcelona,SP1
145,240,1636.558272,Inter Milan,I1
70,76,1633.508785,Chelsea,E0
66,420,1610.155083,Atletico Madrid,SP1


CL data shows a more "reasonable" ranking. Hence it makes sense to use this data with a higher K (to be investigated). We can also see we don't have that many - only ~2000. Hence the weak Elo separation. As an investigation, let's use this to find out the average Elo of each league. 

In [67]:
cl_elo_df.groupby("league")["elo"].mean().sort_values(ascending=False)

league
E0     1598.737133
I1     1549.132860
SP1    1546.349031
P1     1544.543375
D1     1530.460034
F1     1515.282855
N1     1508.691089
G1     1484.612087
B1     1482.128009
T1     1474.693352
SC0    1469.401079
Name: elo, dtype: float64

In [ ]:
# testing different K values for CL matchups, (30, 40, 50)

team_ids = pd.unique(
    matches[["home_team_id", "away_team_id"]].values.ravel()
)

elo_results = {}

for test_k in [30, 40, 50]:

    elo = {team_id: 1500 for team_id in team_ids}

    for _, match in matches.iterrows():

        K = test_k if match["competition"] == "CL" else 20

        S_A = 1 if match["home_goals"] > match["away_goals"] else 0.5 if match["home_goals"] == match["away_goals"] else 0
        S_B = 1 if match["home_goals"] < match["away_goals"] else 0.5 if match["home_goals"] == match["away_goals"] else 0

        current_home_rating = elo[match["home_team_id"]]
        current_away_rating = elo[match["away_team_id"]]

        Q_A = 10 ** (current_home_rating / 400)
        Q_B = 10 ** (current_away_rating / 400)

        expected_home_score = Q_A / (Q_A + Q_B)
        expected_away_score = Q_B / (Q_A + Q_B)

        elo[match["home_team_id"]] = current_home_rating + K * (S_A - expected_home_score)
        elo[match["away_team_id"]] = current_away_rating + K * (S_B - expected_away_score)

    elo_df = pd.DataFrame(
        list(elo.items()),
        columns=["team_id", "elo"]
    )

    elo_df = elo_df.merge(
        teams,
        on="team_id",
        how="left"
    )

    elo_df = elo_df.sort_values("elo", ascending=False)

    elo_results[test_k] = elo_df

display(elo_results[30].head(20))
display(elo_results[40].head(20))
display(elo_results[50].head(20))

,team_id,elo,canonical_name,league
133,156,1923.729137,Bayern Munich,D1
106,70,1886.941748,Arsenal,E0
78,140,1856.396663,Paris Saint-Germain,F1
231,421,1837.056033,Barcelona,SP1
237,440,1811.604052,Real Madrid,SP1
152,359,1795.492998,Porto,P1
136,363,1794.074779,Sporting CP,P1
220,240,1787.964820,Inter Milan,I1
122,87,1778.070375,Manchester City,E0
164,343,1768.954627,Benfica,P1


,team_id,elo,canonical_name,league
133,156,1945.876666,Bayern Munich,D1
106,70,1913.512928,Arsenal,E0
78,140,1886.314908,Paris Saint-Germain,F1
231,421,1849.067471,Barcelona,SP1
237,440,1822.482346,Real Madrid,SP1
136,363,1795.604086,Sporting CP,P1
152,359,1793.006516,Porto,P1
220,240,1786.663632,Inter Milan,I1
122,87,1782.876180,Manchester City,E0
124,310,1760.632274,PSV Eindhoven,N1


,team_id,elo,canonical_name,league
133,156,1965.881098,Bayern Munich,D1
106,70,1937.052027,Arsenal,E0
78,140,1915.160936,Paris Saint-Germain,F1
231,421,1860.510538,Barcelona,SP1
237,440,1832.152648,Real Madrid,SP1
136,363,1802.397139,Sporting CP,P1
152,359,1791.853630,Porto,P1
122,87,1786.389810,Manchester City,E0
220,240,1785.136602,Inter Milan,I1
124,310,1755.391134,PSV Eindhoven,N1


We can now see that the top leagues are rising to the top more consistently, but we're still overrepresented by etams which are stronger in their leagues, but not necessarily on a global stage. Although important not to overfit to "perception", there are some clear issues in this listing. As such. we should implement different starting Elos per league. We must ask the question of how to choose such values fairly.

We need to choose which K is best, and then following that find a better way to interrelate the leagues.

One idea is:
Take an external elo source from start of/pre 2015/2016, and calculate the average elo for each league. We can then scale it and reaverage around 1500: ie


R_league = 1500 + (league Elo - league average)

We cannot use current or future values to initialise elo, so this seems a clean way of doing so, importing only one value per team.

It then makes sense to recompute the multiple K's after this, and find a metric to choose the optimal K for CL games.

We will now get data values from ClubElo

In [ ]:
import soccerdata as sd

clubelo = sd.ClubElo()

elo_2015 = clubelo.read_by_date("2015-06-30")

elo_2015.head()

[08/13/26 15:13:26] INFO     Saving cached data to C:\Users\rohan\soccerdata\data\ClubElo            ]8;id=5782913;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=5782914;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

,rank,country,level,elo,from,to,league
team,,,,,,,
Barcelona,1.0,ESP,1,2087.628174,2015-06-07,2015-07-30,ESP-La Liga
Real Madrid,2.0,ESP,1,2062.575928,2015-06-07,2015-07-30,ESP-La Liga
Bayern,3.0,GER,1,1980.296631,2015-05-24,2015-07-30,GER-Bundesliga
Atletico,4.0,ESP,1,1945.396240,2015-06-07,2015-07-30,ESP-La Liga
Juventus,5.0,ITA,1,1932.656616,2015-06-07,2015-07-30,ITA-Serie A


In [77]:
countries = [
    "ENG",
    "SCO",
    "GER",
    "ESP",
    "ITA",
    "FRA",
    "BEL",
    "NED",
    "POR",
    "TUR",
    "GRE"
]

top_leagues = elo_2015[
    (elo_2015["country"].isin(countries)) &
    (elo_2015["level"] == 1)
]

league_elos = (
    top_leagues
    .groupby("country")["elo"]
    .mean()
    .sort_values(ascending=False)
)

league_elos

country
ESP    1766.790265
GER    1704.133748
ENG    1691.311475
ITA    1661.208600
FRA    1607.575476
POR    1518.692322
TUR    1506.674601
BEL    1443.854294
NED    1440.908542
GRE    1387.999410
SCO    1296.675364
Name: elo, dtype: float64

Using this data, we'll recompute the Elo ratings, with starting values as shown.

In [ ]:
# third pass of computing elos. initialised according to ClubElo data, K = 20 fixed

# conversion dict for league to elo avg
league_to_country = {
    "E0": "ENG",
    "SC0": "SCO",
    "D1": "GER",
    "SP1": "ESP",
    "I1": "ITA",
    "F1": "FRA",
    "B1": "BEL",
    "N1": "NED",
    "P1": "POR",
    "T1": "TUR",
    "G1": "GRE"
}

team_leagues = pd.read_sql_query(
    "SELECT DISTINCT team_id, league FROM team_aliases WHERE source = 'football_data'",
    connection
)

initialised_elo = {}

for _, team in team_leagues.iterrows():

    league = team["league"]
    country = league_to_country[league]

    starting_rating = league_elos.loc[country]

    initialised_elo[team["team_id"]] = starting_rating


K = 20

for _, match in matches.iterrows():

    # for teams outside our 11 domestic leagues
    if match["home_team_id"] not in initialised_elo:
        initialised_elo[match["home_team_id"]] = 1500

    if match["away_team_id"] not in initialised_elo:
        initialised_elo[match["away_team_id"]] = 1500

    S_A = 1 if match["home_goals"] > match["away_goals"] else 0.5 if match["home_goals"] == match["away_goals"] else 0
    S_B = 1 if match["home_goals"] < match["away_goals"] else 0.5 if match["home_goals"] == match["away_goals"] else 0

    current_home_rating = initialised_elo[match["home_team_id"]]
    current_away_rating = initialised_elo[match["away_team_id"]]

    Q_A = 10 ** (current_home_rating / 400)
    Q_B = 10 ** (current_away_rating / 400)

    expected_home_score = Q_A / (Q_A + Q_B)
    expected_away_score = Q_B / (Q_A + Q_B)

    updated_home_rating = current_home_rating + K * (S_A - expected_home_score)
    updated_away_rating = current_away_rating + K * (S_B - expected_away_score)

    initialised_elo[match["home_team_id"]] = updated_home_rating
    initialised_elo[match["away_team_id"]] = updated_away_rating

initialised_elo_df = pd.DataFrame(
    list(initialised_elo.items()),
    columns=["team_id", "elo"]
)

teams = pd.read_sql_query(
    "SELECT team_id, canonical_name FROM teams",
    connection
)

initialised_elo_df = initialised_elo_df.merge(
    teams,
    on="team_id",
    how="left"
)

initialised_elo_df = initialised_elo_df.sort_values("elo", ascending=False)

top_20 = initialised_elo_df.head(20)

top_20

,team_id,elo,canonical_name
93,156,2050.360927,Bayern Munich
262,421,2018.523373,Barcelona
25,70,2008.559136,Arsenal
281,440,1989.862114,Real Madrid
83,140,1953.110627,Paris Saint-Germain
42,87,1933.987410,Manchester City
163,240,1929.515784,Inter Milan
95,158,1893.968345,Borussia Dortmund
261,420,1878.984622,Atletico Madrid
92,155,1865.260673,Bayer Leverkusen


On inspection this is much more effective than our idea of choosing a valid K. Unsurprising given the (somewhat) small dataset, that priors would make a large impact. We should not neglect the increase K measure for CL matches,a s it appeared to optimise our Elo ratings in a positive dimension.

We need a metric by which to choose our best K for CL values. We could use mean square error of the $S$ value and $E$ value (the average difference between the observed result and the expected result from the elo):

$$
\text{MSE} = \frac{1}{n} \sum{(S_{A} - E_{A})^{2}} 
$$

In [ ]:
matches = matches.sort_values("date").reset_index(drop=True)

mse_results = {}

for test_k in [20, 30, 40, 50]:

    initialised_elo = {}

    for _, team in team_leagues.iterrows():

        league = team["league"]
        country = league_to_country[league]
        starting_rating = league_elos.loc[country]

        initialised_elo[team["team_id"]] = starting_rating

    validation_errors = []

    for _, match in matches.iterrows():

        if match["home_team_id"] not in initialised_elo:
            initialised_elo[match["home_team_id"]] = 1500

        if match["away_team_id"] not in initialised_elo:
            initialised_elo[match["away_team_id"]] = 1500

        if match["competition"] == "CL":
            K = test_k
        else:
            K = 20

        if match["home_goals"] > match["away_goals"]:
            S_A = 1
            S_B = 0

        elif match["home_goals"] == match["away_goals"]:
            S_A = 0.5
            S_B = 0.5

        else:
            S_A = 0
            S_B = 1

        current_home_rating = initialised_elo[match["home_team_id"]]
        current_away_rating = initialised_elo[match["away_team_id"]]

        Q_A = 10 ** (current_home_rating / 400)
        Q_B = 10 ** (current_away_rating / 400)

        expected_home_score = Q_A / (Q_A + Q_B)
        expected_away_score = Q_B / (Q_A + Q_B)

        if match["season"] == "2023/24" and match["competition"] == "CL":
            squared_error = (S_A - expected_home_score) ** 2
            validation_errors.append(squared_error)

        updated_home_rating = current_home_rating + K * (S_A - expected_home_score)
        updated_away_rating = current_away_rating + K * (S_B - expected_away_score)

        initialised_elo[match["home_team_id"]] = updated_home_rating
        initialised_elo[match["away_team_id"]] = updated_away_rating

    mse = sum(validation_errors) / len(validation_errors)

    mse_results[test_k] = mse

for k, mse in mse_results.items():
    print(k, mse)

20 0.1453701493706558
30 0.14479113535327903
40 0.14499139734431643
50 0.14559589932275005


We can see that all the MSE's are of negligible difference, hence we proceed witha universial $K = 20$ for simplicity.

We have now generated our calculated for our Elo table. For completeness, the cell below will contain the final Elo table calculation, with a full print of the Elos.

In [85]:
# final Elo calculation: ClubElo league starting values, K = 20

matches = matches.sort_values("date").reset_index(drop=True)

league_to_country = {
    "E0": "ENG",
    "SC0": "SCO",
    "D1": "GER",
    "SP1": "ESP",
    "I1": "ITA",
    "F1": "FRA",
    "B1": "BEL",
    "N1": "NED",
    "P1": "POR",
    "T1": "TUR",
    "G1": "GRE"
}

team_leagues = pd.read_sql_query(
    "SELECT DISTINCT team_id, league FROM team_aliases WHERE source = 'football_data'",
    connection
)

final_elo = {}

for _, team in team_leagues.iterrows():

    league = team["league"]
    country = league_to_country[league]

    starting_rating = league_elos.loc[country]

    final_elo[team["team_id"]] = starting_rating


K = 20

for _, match in matches.iterrows():

    if match["home_team_id"] not in final_elo:
        final_elo[match["home_team_id"]] = 1500

    if match["away_team_id"] not in final_elo:
        final_elo[match["away_team_id"]] = 1500

    if match["home_goals"] > match["away_goals"]:
        S_A = 1
        S_B = 0

    elif match["home_goals"] == match["away_goals"]:
        S_A = 0.5
        S_B = 0.5

    else:
        S_A = 0
        S_B = 1

    current_home_rating = final_elo[match["home_team_id"]]
    current_away_rating = final_elo[match["away_team_id"]]

    Q_A = 10 ** (current_home_rating / 400)
    Q_B = 10 ** (current_away_rating / 400)

    expected_home_score = Q_A / (Q_A + Q_B)
    expected_away_score = Q_B / (Q_A + Q_B)

    updated_home_rating = current_home_rating + K * (S_A - expected_home_score)
    updated_away_rating = current_away_rating + K * (S_B - expected_away_score)

    final_elo[match["home_team_id"]] = updated_home_rating
    final_elo[match["away_team_id"]] = updated_away_rating


final_elo_df = pd.DataFrame(
    list(final_elo.items()),
    columns=["team_id", "elo"]
)

teams = pd.read_sql_query(
    "SELECT team_id, canonical_name FROM teams",
    connection
)

final_elo_df = final_elo_df.merge(
    teams,
    on="team_id",
    how="left"
)

final_elo_df = final_elo_df.sort_values("elo", ascending=False)

display(final_elo_df)

,team_id,elo,canonical_name
93,156,2050.360927,Bayern Munich
262,421,2018.523373,Barcelona
25,70,2008.559136,Arsenal
281,440,1989.862114,Real Madrid
83,140,1953.110627,Paris Saint-Germain
42,87,1933.987410,Manchester City
163,240,1929.515784,Inter Milan
95,158,1893.968345,Borussia Dortmund
261,420,1878.984622,Atletico Madrid
92,155,1865.260673,Bayer Leverkusen
